# TP 04 — Lancer Jupyter et lire un fichier Parquet

**Objectif :** télécharger les trajets des taxis jaunes de New York (janvier 2021), les lire avec pandas et observer leur structure.

Environnement : noyau **Python (mlops)** — Python 3.12, pandas, pyarrow.

## 1. Télécharger les données

Le fichier est enregistré dans `../data`, un dossier ignoré par Git (`data/` dans `.gitignore`) : les données ne sont pas versionnées.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet"
file_path = data_dir / "yellow_tripdata_2021-01.parquet"

if not file_path.exists():
    urlretrieve(url, file_path)

file_path

PosixPath('../data/yellow_tripdata_2021-01.parquet')

## 2. Lire le fichier Parquet

In [2]:
import pandas as pd

df = pd.read_parquet(file_path)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1.0,2.10,1.0,N,142,43,2,8.0,3.0,0.5,0.00,0.0,0.3,11.80,2.5,NaN
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1.0,0.20,1.0,N,238,151,2,3.0,0.5,0.5,0.00,0.0,0.3,4.30,0.0,NaN
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1.0,14.70,1.0,N,132,165,1,42.0,0.5,0.5,8.65,0.0,0.3,51.95,0.0,NaN
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0.0,10.60,1.0,N,138,132,1,29.0,0.5,0.5,6.05,0.0,0.3,36.35,0.0,NaN
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1.0,4.94,1.0,N,68,33,1,16.5,0.5,0.5,4.06,0.0,0.3,24.36,2.5,NaN


In [3]:
df.shape

(1369769, 19)

In [4]:
df.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
dtype: object

Les colonnes `tpep_pickup_datetime` et `tpep_dropoff_datetime` sont déjà lues comme des dates : le format Parquet conserve le type de chaque colonne, alors qu'un fichier CSV demanderait souvent une conversion.

In [5]:
df[["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]].head()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID
0,2021-01-01 00:30:10,2021-01-01 00:36:12,142,43
1,2021-01-01 00:51:20,2021-01-01 00:52:19,238,151
2,2021-01-01 00:43:30,2021-01-01 01:11:06,132,165
3,2021-01-01 00:15:48,2021-01-01 00:31:01,138,132
4,2021-01-01 00:31:49,2021-01-01 00:48:21,68,33


## 3. Exercice : lire plusieurs mois

On ajoute le fichier de février 2021, puis on lit automatiquement tous les fichiers `yellow_tripdata_*.parquet` présents dans `data`.

In [6]:
urls = {
    "2021-01": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet",
    "2021-02": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-02.parquet",
}

for month, url in urls.items():
    destination = data_dir / f"yellow_tripdata_{month}.parquet"
    if not destination.exists():
        urlretrieve(url, destination)

In [7]:
files = sorted(data_dir.glob("yellow_tripdata_*.parquet"))
frames = []

for parquet_file in files:
    month_df = pd.read_parquet(parquet_file)                       # lire le fichier
    month = parquet_file.stem.replace("yellow_tripdata_", "")     # ex. "2021-01"
    month_df["source_month"] = month                               # mois d'origine
    frames.append(month_df)

trips = pd.concat(frames, ignore_index=True)
trips.groupby("source_month").size()

source_month
2021-01    1369769
2021-02    1371709
dtype: int64

### Vérification

Le total de `trips` doit être égal à la somme des lignes de chaque fichier. Le code fonctionne aussi si un nouveau fichier mensuel est ajouté dans `data`.

In [8]:
rows_per_file = {f.name: len(pd.read_parquet(f)) for f in files}
assert len(trips) == sum(rows_per_file.values())
print(rows_per_file)
print("Total :", len(trips), "lignes — vérification réussie")

{'yellow_tripdata_2021-01.parquet': 1369769, 'yellow_tripdata_2021-02.parquet': 1371709}
Total : 2741478 lignes — vérification réussie
